# Presentation figures

Regenerates the summary figures from the result JSONs written by each benchmark script. Save location: `figs/presentation/`.

Run with `conda run -n rdkit_env jupyter nbconvert --execute`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path('..').resolve().parent
OUT = REPO / 'figs' / 'presentation'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 160, 'savefig.bbox': 'tight', 'font.size': 11})

def load_summary(path):
    with open(path) as f:
        return json.load(f)

def test_metrics(summary):
    th = summary.get('training_history', {})
    return th.get('test_metrics') or summary.get('test_metrics') or {}

print('REPO:', REPO)
print('OUT:', OUT)

## Load all result summaries into a dataframe

In [ ]:
rows = []

def add(dataset, split, model, summary, n_override=None):
    tm = test_metrics(summary)
    rows.append({
        'dataset': dataset,
        'split': split,
        'model': model,
        'roc_auc': tm.get('roc_auc'),
        'avg_precision': tm.get('avg_precision'),
        'pearson': tm.get('pearson'),
        'spearman': tm.get('spearman'),
        'r2': tm.get('r2'),
        'rmse': tm.get('rmse'),
        'n_test': n_override if n_override is not None else summary.get('training_history', {}).get('n_test_samples'),
    })

# PLATE-VS hard 0p7 classification
hard_dir = REPO / 'benchmarks' / '02_training' / 'trained_models'
for model, fname in [('RF', 'random_forest'), ('GBM', 'gradient_boosting'), ('SVM', 'svm')]:
    p = hard_dir / f'{fname}_training_summary.json'
    if p.exists():
        add('PLATE-VS', 'hard 0p7', model, load_summary(p))

# GNINA on PLATE-VS hard 0p7 — report per-target MEAN ROC-AUC (minAff scorer)
# computed from per-target JSONs. This is the standard VS summary statistic
# (unweighted mean across targets), not the pooled ROC-AUC.
dmdir = REPO / 'benchmarks' / '04_docking' / 'results' / 'docking_metrics'
_gnina_per_target = []
for p in sorted(dmdir.glob('*_docking_metrics.json')):
    if p.name.startswith('all_targets'):
        continue
    d = load_summary(p)
    if d.get('status') != 'ok':
        continue
    minf = d.get('minimizedAffinity', {})
    if 'roc_auc' in minf:
        _gnina_per_target.append(minf['roc_auc'])
if _gnina_per_target:
    rows.append({
        'dataset': 'PLATE-VS', 'split': 'hard 0p7', 'model': 'GNINA',
        'roc_auc': float(np.mean(_gnina_per_target)),
        'avg_precision': None, 'pearson': None, 'spearman': None, 'r2': None, 'rmse': None,
        'n_test': len(_gnina_per_target),  # number of targets
    })
    print(f"GNINA per-target mean ROC-AUC (minAff): {np.mean(_gnina_per_target):.3f}  (n_targets={len(_gnina_per_target)})")

# PLATE-VS soft 0p7 classification
soft_dir = REPO / 'trained_models' / 'soft_split_classification'
for model, fname in [('RF', 'random_forest'), ('GBM', 'gradient_boosting'), ('SVM', 'svm')]:
    p = soft_dir / f'{fname}_training_summary.json'
    if p.exists():
        add('PLATE-VS', 'soft 0p7', model, load_summary(p))

# PLATE-VS soft 0p7 regression
reg_dir = REPO / 'trained_models' / 'regression'
for model, fname in [('RF', 'random_forest_regressor'), ('GBM', 'gradient_boosting_regressor'), ('SVM', 'svm_regressor')]:
    p = reg_dir / f'{fname}_training_summary.json'
    if p.exists():
        add('PLATE-VS', 'soft 0p7 (reg)', model, load_summary(p))

# PDBbind CASF-2016 regression — classical with protein embeddings
pdb_dir = REPO / 'benchmarks' / '05_pdbbind_comparison' / 'results' / 'classical_with_prot_emb'
for model, fname in [('RF', 'random_forest'), ('GBM', 'gradient_boosting'), ('SVM', 'svm')]:
    p = pdb_dir / f'{fname}_pdbbind_casf2016_training_summary.json'
    if p.exists():
        add('PDBbind', 'CASF-2016', model, load_summary(p))

# GEMS
p = REPO / 'benchmarks' / '05_pdbbind_comparison' / 'results' / 'gems' / 'gems_casf2016_training_summary.json'
if p.exists():
    add('PDBbind', 'CASF-2016', 'GEMS', load_summary(p))

# Dual-encoder ensemble
p = REPO / 'benchmarks' / '05_pdbbind_comparison' / 'results' / 'dual_encoder_ensemble_training_summary.json'
if p.exists():
    add('PDBbind', 'CASF-2016', 'DualEnc (5f)', load_summary(p))

# GNINA on 20-complex CASF-2016 subset (session-scale run)
p = REPO / 'benchmarks' / '05_pdbbind_comparison' / 'results' / 'gnina_pdbbind_casf2016_training_summary.json'
if p.exists():
    add('PDBbind', 'CASF-2016', 'GNINA (n=20)', load_summary(p), n_override=20)

df = pd.DataFrame(rows)
df

## Fig 1 — PLATE-VS hard 0p7 classification

In [ ]:
sub = df[(df.dataset == 'PLATE-VS') & (df.split == 'hard 0p7')].set_index('model').reindex(['RF', 'GBM', 'SVM', 'GNINA'])
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(sub.index, sub.roc_auc, color=['#4c72b0'] * 3 + ['#c44e52'])
ax.axhline(0.5, color='gray', ls='--', lw=0.8, label='random')
ax.set_ylim(0, 0.8)
ax.set_ylabel('Test ROC-AUC')
ax.set_title('PLATE-VS hard 0p7 — classification')
# Footnote about the GNINA statistic
ax.text(3, sub.roc_auc.iloc[3] + 0.07, '(per-target mean,\n minAff, n=15)',
        ha='center', fontsize=8, color='dimgray', style='italic')
ax.legend(loc='upper left')
for bar, v in zip(bars, sub.roc_auc):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
fig.savefig(OUT / 'fig1_plate_vs_hard_classification.png')
plt.show()
print(sub[['roc_auc', 'avg_precision', 'n_test']])

## Fig 2 — PLATE-VS soft 0p7 classification

In [ ]:
sub = df[(df.dataset == 'PLATE-VS') & (df.split == 'soft 0p7')].set_index('model').reindex(['RF', 'GBM', 'SVM'])
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(sub.index, sub.roc_auc, color='#55a868')
ax.axhline(0.5, color='gray', ls='--', lw=0.8, label='random')
ax.set_ylim(0, 0.8)
ax.set_ylabel('Test ROC-AUC')
ax.set_title('PLATE-VS soft 0p7 — classification')
ax.legend(loc='upper left')
for bar, v in zip(bars, sub.roc_auc):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
fig.savefig(OUT / 'fig2_plate_vs_soft_classification.png')
plt.show()

## Fig 3 — Hard vs soft grouped comparison (classification)

In [ ]:
models = ['RF', 'GBM', 'SVM']
hard = [df[(df.dataset == 'PLATE-VS') & (df.split == 'hard 0p7') & (df.model == m)].roc_auc.values[0] for m in models]
soft = [df[(df.dataset == 'PLATE-VS') & (df.split == 'soft 0p7') & (df.model == m)].roc_auc.values[0] for m in models]
x = np.arange(len(models))
w = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - w / 2, hard, w, label='hard 2D split', color='#c44e52')
ax.bar(x + w / 2, soft, w, label='soft split', color='#55a868')
ax.axhline(0.5, color='gray', ls='--', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('Test ROC-AUC')
ax.set_title('PLATE-VS: hard vs soft split — generalization gap')
ax.set_ylim(0, 0.7)
ax.legend()
for xi, (h, s) in enumerate(zip(hard, soft)):
    ax.text(xi - w / 2, h + 0.01, f'{h:.3f}', ha='center', fontsize=9)
    ax.text(xi + w / 2, s + 0.01, f'{s:.3f}', ha='center', fontsize=9)
fig.savefig(OUT / 'fig3_hard_vs_soft_generalization.png')
plt.show()
print('gap (soft - hard):', [round(s - h, 3) for h, s in zip(hard, soft)])

## Fig 4 — PDBbind CASF-2016 regression (Pearson R)

In [ ]:
order = ['RF', 'GBM', 'SVM', 'DualEnc (5f)', 'GNINA (n=20)', 'GEMS']
sub = df[(df.dataset == 'PDBbind')].set_index('model').reindex(order)
# Per-fold spread for dual-encoder
fold_R = []
for i in range(5):
    p = REPO / 'benchmarks' / '06_binding_affinity_model' / 'results' / f'dual_encoder_f{i}_casf2016_training_summary.json'
    if p.exists():
        fold_R.append(test_metrics(load_summary(p))['pearson'])
print(f'Per-fold Pearson R: {fold_R}  mean={np.mean(fold_R):.3f}  std={np.std(fold_R):.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4c72b0'] * 3 + ['#8172b2', '#dd8452', '#c44e52']
bars = ax.bar(sub.index, sub.pearson, color=colors)
ax.set_ylabel('Test Pearson R')
ax.set_title('PDBbind CleanSplit CASF-2016 — regression')
ax.set_ylim(0, 1.0)
# Dual-encoder per-fold error bar
if fold_R:
    dual_idx = list(sub.index).index('DualEnc (5f)')
    ax.errorbar(dual_idx, np.mean(fold_R), yerr=np.std(fold_R), fmt='none', ecolor='black', capsize=4)
for bar, v in zip(bars, sub.pearson):
    if v is not None and not pd.isna(v):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.015, f'{v:.3f}', ha='center', fontsize=10)
ax.tick_params(axis='x', rotation=20)
# Caveat annotation for GNINA subset
gnina_idx = list(sub.index).index('GNINA (n=20)')
ax.annotate('stratified n=20\n(of 285)', xy=(gnina_idx, sub.pearson.iloc[gnina_idx]),
            xytext=(gnina_idx, 0.35), fontsize=8, ha='center', color='dimgray',
            arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
fig.savefig(OUT / 'fig4_pdbbind_casf2016_regression.png')
plt.show()
print(sub[['pearson', 'spearman', 'r2', 'rmse', 'n_test']])

## Fig 5 — GNINA per-target ROC-AUC distribution on PLATE-VS

Two GNINA scoring functions: `CNN_VS` (CNN virtual-screen score) and `minimizedAffinity` (Vina-style scored affinity).

In [ ]:
dmdir = REPO / 'benchmarks' / '04_docking' / 'results' / 'docking_metrics'
cnn_aucs, min_aucs, targets = [], [], []
for p in sorted(dmdir.glob('*_docking_metrics.json')):
    if p.name.startswith('all_targets'):
        continue
    d = load_summary(p)
    if d.get('status') != 'ok':
        continue
    cnn = d.get('CNN_VS', {})
    minf = d.get('minimizedAffinity', {})
    if 'roc_auc' not in cnn or 'roc_auc' not in minf:
        continue
    targets.append(d['uniprot'])
    cnn_aucs.append(cnn['roc_auc'])
    min_aucs.append(minf['roc_auc'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([cnn_aucs, min_aucs], labels=['CNN_VS', 'minimizedAffinity'], showmeans=True,
           medianprops=dict(color='black'), meanprops=dict(marker='D', markerfacecolor='red', markersize=6))
for i, data in enumerate([cnn_aucs, min_aucs], start=1):
    ax.scatter(np.full(len(data), i) + np.random.uniform(-0.05, 0.05, len(data)), data, alpha=0.5, s=30, color='#4c72b0')
ax.axhline(0.5, color='gray', ls='--', lw=0.8)
ax.set_ylabel('Per-target ROC-AUC')
ax.set_title(f'GNINA per-target ROC-AUC on PLATE-VS ({len(targets)} targets)')
ax.set_ylim(0, 1.0)
fig.savefig(OUT / 'fig5_gnina_per_target_auc.png')
plt.show()
print(f'CNN_VS   mean={np.mean(cnn_aucs):.3f}  median={np.median(cnn_aucs):.3f}')
print(f'minAff   mean={np.mean(min_aucs):.3f}  median={np.median(min_aucs):.3f}')
print(f'targets with AUC ({len(targets)}/15): {targets}')


## Summary — all saved figures

In [ ]:
for p in sorted(OUT.glob('*.png')):
    print(p.relative_to(REPO))

## Fig 6 — PLATE-VS soft 0p7 with projected dual-encoder & GEMS

**Measured**: RF / GBM / SVM — solid bars, real training-summary numbers.

**Projected** (hatched, with ± error bars) — these are reasoning-based estimates, **not experiments**:

- **Dual-encoder** 0.72 ± 0.04. Hard-split 2-epoch W&B log showed ≈ 0.80 but that run was heavily under-trained and its per-target AUC pipeline was broken (fixed today in `07_plate_vs_dl/evaluation.py`). A properly-evaluated, more fully trained model on soft split likely lands lower — around the low-0.7s is our current best guess.
- **GEMS (pretrained)** 0.64 ± 0.08. No direct measurement. GEMS was trained on PDBbind crystals for pK regression; repurposing its pK scores as a VS ranker on ChEMBL + DeepCoy is heavily out-of-distribution. Expect modest discrimination — clearly above random, but well below a PLATE-VS task-tuned model. Wide band reflects real uncertainty.

These bars are there to illustrate **expected ranges**, not results. Disclose them as projections when presenting.

In [ ]:
# Measured soft-split values (from df loaded above)
measured = df[(df.dataset == 'PLATE-VS') & (df.split == 'soft 0p7')].set_index('model').reindex(['RF', 'GBM', 'SVM'])

# Projected values (reasoning-based — see markdown cell above)
projected = [
    ('DualEnc\n(ours)',  0.72, 0.04),
    ('GEMS\n(pretrained)', 0.64, 0.08),
]

labels   = list(measured.index) + [p[0] for p in projected]
heights  = list(measured.roc_auc.values) + [p[1] for p in projected]
errs     = [0] * len(measured) + [p[2] for p in projected]
is_proj  = [False] * len(measured) + [True, True]

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(labels))
for xi, (h, e, proj) in enumerate(zip(heights, errs, is_proj)):
    if proj:
        bar = ax.bar(xi, h, color='#8172b2', hatch='///', edgecolor='white', linewidth=0.8,
                     yerr=e, capsize=5, error_kw=dict(ecolor='black', lw=1.2))
    else:
        bar = ax.bar(xi, h, color='#55a868')

ax.axhline(0.5, color='gray', ls='--', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Test ROC-AUC')
ax.set_title('PLATE-VS soft 0p7 — measured + projected')
ax.set_ylim(0, 1.0)

# Value labels
for xi, (h, e, proj) in enumerate(zip(heights, errs, is_proj)):
    if proj:
        ax.text(xi, h + e + 0.025, f'{h:.2f} ± {e:.2f}', ha='center', fontsize=9, style='italic')
    else:
        ax.text(xi, h + 0.015, f'{h:.3f}', ha='center', fontsize=10)

# Legend (custom patches)
from matplotlib.patches import Patch
legend = [
    Patch(facecolor='#55a868', label='measured'),
    Patch(facecolor='#8172b2', hatch='///', edgecolor='white', label='projected (see caption)'),
]
ax.legend(handles=legend, loc='upper left', framealpha=0.95)

fig.savefig(OUT / 'fig6_plate_vs_soft_with_projections.png')
plt.show()
print('Projected values (mean ± range):')
for name, mean, err in projected:
    print(f'  {name.replace(chr(10), " ")}: {mean:.2f} ± {err:.2f}')